# Lesson 3.2: The Binomial Distribution

Use the binomial distribution for the lesson's drug that works for 70% of patients (10 patients): exactly, at most and at least, plus mean, SD and a simulation check.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A drug works for 70% of patients. A clinic gives it to 10 patients. The doctor asks: how likely is it that exactly 7 respond, that at most 6 respond, and that at least 9 respond? What is the average number of responders and how much does it vary from clinic to clinic? Can the binomial distribution answer all of this, and does a simulation agree?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Exactly 7 respond

binom.pmf(k, n, p) gives P(X = k), one bar of the distribution. We also compute the lesson's formula C(n, k) x p^k x (1 - p)^(n - k) by hand to see that they agree.

In [1]:
import math  # Load math for the combination function.
import numpy as np  # Load NumPy, which we will need for the simulation later.
from scipy.stats import binom  # Load the binomial distribution from SciPy.

n, p = 10, 0.7  # Store the number of patients n and the chance of response p.

by_hand = math.comb(n, 7) * p**7 * (1 - p)**3  # Compute the lesson's formula by hand: choose which 7 respond, then multiply their chances and the 3 failures' chances.
print(f'formula   : {by_hand:.4f}')  # Print the hand formula.
print(f'binom.pmf : {binom.pmf(7, n, p):.4f}')  # Print SciPy's pmf, which gives P(X = 7) directly. The two numbers should match.

formula   : 0.2668
binom.pmf : 0.2668


**What this means.** Both ways give P(X = 7) = 0.2668. About 27% of the time, exactly 7 of 10 patients respond. The hand formula and the SciPy function agree, so we can trust the function for harder questions.

## Step 2: At most, more than, at least

cdf(k) is P(X <= k), 'at most k'. sf(k) is the survival function P(X > k), 'more than k'. So 'at least 9' is sf(8), which is the same as 1 minus cdf(8).

In [2]:
print(f'P(X <= 6)  at most 6      = {binom.cdf(6, n, p):.4f}')  # P(at most 6): cdf(k) adds the probabilities of 0 up to k.
print(f'P(X > 8)   more than 8      = {binom.sf(8, n, p):.4f}')  # P(more than 8): sf(k) is the survival function, the chance of strictly more than k.
print(f'P(X >= 9)  at least 9       = {1 - binom.cdf(8, n, p):.4f}')  # P(at least 9) is 1 minus the chance of 8 or fewer.
print(f'P(9) + P(10) added by hand  = {binom.pmf(9, n, p) + binom.pmf(10, n, p):.4f}')  # Check by adding P(9) and P(10) by hand. Three methods should agree.

P(X <= 6)  at most 6      = 0.3504
P(X > 8)   more than 8      = 0.1493
P(X >= 9)  at least 9       = 0.1493
P(9) + P(10) added by hand  = 0.1493


**What this means.** P(X <= 6) = 0.3504, so a bit over a third of clinics see 6 or fewer responders. P(X > 8) and P(X >= 9) are both 0.1493, and P(9) + P(10) is also 0.1493. Rather few clinics (about 15%) would see 9 or more.

## Step 3: Mean and standard deviation

The shortcuts are mean = n x p and SD = square root of n x p x (1 - p). SciPy's stats method returns the same values from the distribution itself.

In [3]:
print(f'mean = n*p = {n * p:.1f}')  # The mean of a binomial is n x p.
print(f'variance   = {n * p * (1 - p):.2f}')  # The variance is n x p x (1 - p).
print(f'SD         = {math.sqrt(n * p * (1 - p)):.3f}')  # The SD is the square root of the variance, back in the units of patients.

m, v = binom.stats(n, p)  # Ask SciPy for the mean and variance of the distribution itself.
print(f'scipy      : mean {float(m):.1f}, variance {float(v):.2f}')  # Print what SciPy returns, to check against our shortcuts.

mean = n*p = 7.0
variance   = 2.10
SD         = 1.449
scipy      : mean 7.0, variance 2.10


**What this means.** The mean is 7.0 responders, the variance is 2.10 and the SD is 1.449. SciPy returns the same mean and variance. A typical clinic sees 7 responders, give or take about 1.4.

## Step 4: Check by simulation

Run the 10-patient trial 100,000 times with a seeded generator and count how many responders each trial has. The simulated shares should be close to the exact probabilities.

In [4]:
rng = np.random.default_rng(42)  # Create a seeded generator so the simulation repeats.
responders = rng.binomial(n, p, size=100_000)  # Run the 10-patient trial 100,000 times and record each trial's number of responders.

print(f'simulated P(X = 7)  = {(responders == 7).mean():.4f}  (exact {binom.pmf(7, n, p):.4f})')  # Print the simulated chance of exactly 7 next to the exact value.
print(f'simulated P(X >= 9) = {(responders >= 9).mean():.4f}  (exact {binom.sf(8, n, p):.4f})')  # Print the simulated chance of 9 or more next to the exact value.
print(f'simulated mean {responders.mean():.3f}, SD {responders.std():.3f}')  # Print the simulated mean and SD.

simulated P(X = 7)  = 0.2679  (exact 0.2668)
simulated P(X >= 9) = 0.1485  (exact 0.1493)
simulated mean 6.997, SD 1.450


**What this means.** The simulation gave P(X = 7) = 0.2679 (exact 0.2668), P(X >= 9) = 0.1485 (exact 0.1493), a mean of 6.997 and an SD of 1.450. All are close to the exact answers, which shows the formulas describe what really happens.

## So, what is the answer?

With 10 patients and a 70% response rate, exactly 7 respond about 26.7% of the time, 6 or fewer respond 35.0% of the time, and 9 or more respond 14.9% of the time. The average is 7 responders with an SD of 1.45. The caution: the binomial assumes every patient has the same 70% chance independently, which real patients may not.

## Now you try

A multiple-choice quiz has 12 questions, each with 4 options and only one correct. A student who has not studied guesses every answer, so each guess is right with probability 0.25. The teacher wants to know how likely a guesser is to do well or very badly.

**Data:** n = 12 questions, p = 0.25 chance of a correct guess on each. For the simulation use the seed 2 and 100,000 quizzes.

1. Find the chance of exactly 5 correct answers.
2. Find the chance of 8 or more correct, and the chance of 3 or fewer.
3. Find the mean and SD of the score.
4. Simulate 100,000 quizzes and compare the chance of 8 or more.

*Hint:* This is a binomial with n = 12 and p = 0.25. Use binom.pmf for exactly, binom.sf(7, ...) for 8 or more and binom.cdf(3, ...) for 3 or fewer.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
from scipy.stats import binom
import numpy as np

n, p = 12, 0.25
print(f'P(X = 5)  = {binom.pmf(5, n, p):.4f}')
print(f'P(X >= 8) = {binom.sf(7, n, p):.4f}')
print(f'P(X <= 3) = {binom.cdf(3, n, p):.4f}')
print(f'mean = {n * p}, sd = {np.sqrt(n * p * (1 - p)):.3f}')

rng = np.random.default_rng(2)
guesses = rng.binomial(n, p, size=100_000)
print(f'simulated P(X >= 8) = {(guesses >= 8).mean():.5f}')
print(f'simulated mean = {guesses.mean():.3f}')

P(X = 5)  = 0.1032
P(X >= 8) = 0.0028
P(X <= 3) = 0.6488
mean = 3.0, sd = 1.500
simulated P(X >= 8) = 0.00267
simulated mean = 3.002


**Answer.** P(X = 5) = 0.1032, P(X >= 8) = 0.0028 (fewer than 3 guessers in 1,000) and P(X <= 3) = 0.6488. The mean is 3.0 correct with an SD of 1.500. The simulation gave P(X >= 8) = 0.00267 and a mean of 3.002, in line with the exact values. Guessing almost never passes a 12-question quiz.

## The same question in R, SPSS and Stata

**R**

```r
n <- 10; p <- 0.7  # Store the number of patients and the response chance.
dbinom(7, n, p)                       # exactly 7  # dbinom gives P(X = 7).
pbinom(6, n, p)                       # at most 6  # pbinom gives P(X <= 6).
pbinom(8, n, p, lower.tail = FALSE)   # more than 8 = at least 9  # lower.tail = FALSE gives the chance of more than 8, which is at least 9.
n * p  # The mean is n x p.
sqrt(n * p * (1 - p))  # The SD is the square root of n x p x (1 - p).

set.seed(42)  # Fix the seed.
x <- rbinom(100000, n, p)  # Simulate 100,000 trials of 10 patients.
mean(x == 7)  # Share of simulated trials with exactly 7 responders.
```

How to read it: The output shows 0.2668 for exactly 7, 0.3504 for at most 6, 0.1493 for at least 9, then the mean 7 and the SD 1.449. The last number is the simulated share of 7s, close to 0.2668.

**SPSS** (Transform > Compute Variable (Function group: PDF & Noncentral PDF, CDF & Noncentral CDF, Random Numbers))

```
DATA LIST FREE / dummy.
BEGIN DATA
1
END DATA.
COMPUTE exact7 = PDF.BINOM(7, 10, 0.7).
COMPUTE atmost6 = CDF.BINOM(6, 10, 0.7).
COMPUTE atleast9 = 1 - CDF.BINOM(8, 10, 0.7).
COMPUTE mean_x = 10 * 0.7.
COMPUTE sd_x = SQRT(10 * 0.7 * 0.3).
FORMATS exact7 atmost6 atleast9 sd_x (F8.4).
LIST VARIABLES=exact7 atmost6 atleast9 mean_x sd_x.
```

How to read it: In the LIST output, exact7 is 0.2668, atmost6 is 0.3504, atleast9 is 0.1493, mean_x is 7 and sd_x is 1.4491. They match Python.

**Stata**

```
display binomialp(10, 7, 0.7)
display binomial(10, 6, 0.7)
display binomialtail(10, 9, 0.7)
display 10 * 0.7
display sqrt(10 * 0.7 * 0.3)

clear
set seed 42
set obs 100000
gen x = rbinomial(10, 0.7)
gen is7 = (x == 7)
summarize is7
```

How to read it: The first five lines print .2668, .3504, .1493, 7 and 1.4491. The last summarize gives the simulated share of exactly-7 trials, which is close to .2668 but not identical to Python's 0.2679.